# LangChain: Memory (Groq Llama 3.1)

## Outline
* ConversationBufferMemory - Stores entire conversation
* ConversationBufferWindowMemory - Stores last K exchanges
* ConversationTokenBufferMemory - Limits memory by token count
* ConversationSummaryMemory - Uses LLM to summarize conversation

**Model Used:** Groq Llama-3.1-8b-instant


In [ ]:
# Cell 2: Install Required Packages
!pip install langchain langchain-groq python-dotenv -q

In [ ]:
# Cell 3: Import Libraries and Load Environment

import os
import warnings
from dotenv import load_dotenv, find_dotenv

# Load environment variables
load_dotenv(find_dotenv())

# Suppress warnings
warnings.filterwarnings('ignore')

# Verify API key
groq_api_key = os.environ.get('GROQ_API_KEY')
if not groq_api_key:
    raise ValueError("GROQ_API_KEY not found in .env file!")
    
print("✅ Environment loaded successfully")

In [ ]:
# Cell 4: Initialize Groq LLM

from langchain_groq import ChatGroq

# Set model
llm_model = "llama-3.1-8b-instant"

# Initialize LLM with temperature=0 for consistent outputs
llm = ChatGroq(
    temperature=0.0,
    model=llm_model,
    groq_api_key=groq_api_key
)

print(f"✅ Model initialized: {llm_model}")


In [ ]:
# Cell 5: ConversationBufferMemory - Basic Setup

from langchain.chains import ConversationChain
from langchain.memory import ConversationBufferMemory

# Create memory and conversation chain
memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm=llm, 
    memory=memory,
    verbose=True  # Set to True to see the prompts
)

print("✅ ConversationChain created with ConversationBufferMemory")


In [ ]:
# Cell 6: First Conversation - Introduce Yourself

response = conversation.predict(input="Hi, my name is Andrew")
print(response)


In [ ]:
# Cell 7: Second Conversation - Ask a Question

response = conversation.predict(input="What is 1+1?")
print(response)


In [ ]:
# Cell 8: Third Conversation - Test Memory

# The LLM should remember your name from Cell 6
response = conversation.predict(input="What is my name?")
print(response)


In [ ]:
# Cell 9: Inspect Memory Buffer

# View the raw conversation history stored in memory
print("Memory Buffer Contents:")
print("="*60)
print(memory.buffer)


In [ ]:
# Cell 10: Load Memory Variables

# Alternative way to view memory
memory_variables = memory.load_memory_variables({})
print("Memory Variables:")
print("="*60)
print(memory_variables)


In [ ]:
# Cell 11: Manual Memory Management

# Create fresh memory
memory = ConversationBufferMemory()

# Manually add conversations
memory.save_context({"input": "Hi"}, {"output": "What's up"})

print("After first exchange:")
print(memory.buffer)


In [ ]:
# Cell 12: Add More to Memory

# Add another exchange
memory.save_context(
    {"input": "Not much, just hanging"}, 
    {"output": "Cool"}
)

# View updated memory
print("After second exchange:")
print(memory.load_memory_variables({}))


## ConversationBufferWindowMemory

This memory type only keeps the last **K** conversational exchanges. It's useful when you want to remember recent context without unlimited memory growth.


In [ ]:
# Cell 14: Setup Window Memory (k=1)

from langchain.memory import ConversationBufferWindowMemory

# Create memory that only remembers last 1 exchange
memory = ConversationBufferWindowMemory(k=1)

# Add two exchanges
memory.save_context({"input": "Hi"}, {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"}, {"output": "Cool"})

# Only the most recent exchange is kept
print("Window Memory (k=1):")
print(memory.load_memory_variables({}))


In [ ]:
# Cell 15: Test Window Memory in Conversation

# Create conversation with window memory
llm = ChatGroq(temperature=0.0, model=llm_model, groq_api_key=groq_api_key)
memory = ConversationBufferWindowMemory(k=1)
conversation = ConversationChain(
    llm=llm, 
    memory=memory,
    verbose=False
)

# First exchange
print("Exchange 1:")
print(conversation.predict(input="Hi, my name is Andrew"))
print()


In [ ]:
# Cell 16: Continue Window Memory Conversation

# Second exchange
print("Exchange 2:")
print(conversation.predict(input="What is 1+1?"))
print()


In [ ]:
# Cell 17: Test Window Memory Limitation

# Third exchange - should NOT remember the name from Exchange 1
# because k=1 only keeps the last exchange
print("Exchange 3 (Testing Memory):")
response = conversation.predict(input="What is my name?")
print(response)
print("\n⚠️ Notice: It doesn't remember your name because k=1 dropped that exchange!")


## ConversationTokenBufferMemory

This memory type limits memory based on **token count** rather than number of exchanges. This is useful for controlling LLM API costs since most providers charge per token.


In [ ]:
# Cell 19: Setup Token Buffer Memory

from langchain.memory import ConversationTokenBufferMemory

# Create memory with 50 token limit
memory = ConversationTokenBufferMemory(
    llm=llm, 
    max_token_limit=50
)

# Add short exchanges (ABC pattern for easy tracking)
memory.save_context({"input": "AI is what?!"}, {"output": "Amazing!"})
memory.save_context({"input": "Backpropagation is what?"}, {"output": "Beautiful!"})
memory.save_context({"input": "Chatbots are what?"}, {"output": "Charming!"})

print("Token Buffer Memory (max 50 tokens):")
print(memory.load_memory_variables({}))


In [ ]:
# Cell 20: Test Token Limit Effect

# Increase token limit to 100 to see difference
memory = ConversationTokenBufferMemory(llm=llm, max_token_limit=100)

memory.save_context({"input": "AI is what?!"}, {"output": "Amazing!"})
memory.save_context({"input": "Backpropagation is what?"}, {"output": "Beautiful!"})
memory.save_context({"input": "Chatbots are what?"}, {"output": "Charming!"})

print("Token Buffer Memory (max 100 tokens):")
print(memory.load_memory_variables({}))
print("\n✅ More tokens = more conversation history retained")


## ConversationSummaryBufferMemory

Instead of dropping old conversations, this memory type uses an **LLM to summarize** past exchanges. This keeps context while managing memory size intelligently.


In [ ]:
# Cell 22: Create Long Schedule Text

# Create a detailed schedule string
schedule = """There is a meeting at 8am with your product team. \
You will need your powerpoint presentation prepared. \
9am-12pm have time to work on your LangChain \
project which will go quickly because Langchain is such a powerful tool. \
At Noon, lunch at the italian restaurant with a customer who is driving \
from over an hour away to meet you to understand the latest in AI. \
Be sure to bring your laptop to show the latest LLM demo."""

print("Schedule to be used in conversation:")
print("="*60)
print(schedule)


In [ ]:
# Cell 23: Setup Summary Buffer Memory (High Limit)

from langchain.memory import ConversationSummaryBufferMemory

# Create memory with 400 token limit (high enough to store everything)
memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=400)

# Add conversation about schedule
memory.save_context({"input": "Hello"}, {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"}, {"output": "Cool"})
memory.save_context(
    {"input": "What is on the schedule today?"}, 
    {"output": f"{schedule}"}
)

print("Summary Buffer Memory (400 tokens - stores full text):")
print(memory.load_memory_variables({}))


In [ ]:
# Cell 24: Setup Summary Buffer Memory (Low Limit)

# Reduce token limit to 100 - this will trigger summarization
memory = ConversationSummaryBufferMemory(llm=llm, max_token_limit=100)

memory.save_context({"input": "Hello"}, {"output": "What's up"})
memory.save_context({"input": "Not much, just hanging"}, {"output": "Cool"})
memory.save_context(
    {"input": "What is on the schedule today?"}, 
    {"output": f"{schedule}"}
)

print("Summary Buffer Memory (100 tokens - triggers summarization):")
print("="*60)
print(memory.load_memory_variables({}))
print("\n✅ Notice: The LLM created a summary of the conversation!")


In [ ]:
# Cell 25: Use Summary Memory in Conversation

# Create conversation chain with summary memory
conversation = ConversationChain(
    llm=llm, 
    memory=memory,
    verbose=True
)

# Ask a question based on the schedule
print("Asking for demo suggestions:")
print("="*60)
response = conversation.predict(input="What would be a good demo to show?")
print(response)


In [ ]:
# Cell 26: Inspect Updated Memory

# View how the memory has been updated after the new exchange
print("Updated Memory After Demo Question:")
print("="*60)
print(memory.load_memory_variables({}))
print("\n✅ The new exchange was incorporated into the summary!")


## Summary of Memory Types

| Memory Type | Best For | Limitation Method |
|-------------|----------|-------------------|
| **ConversationBufferMemory** | Short conversations | None (stores everything) |
| **ConversationBufferWindowMemory** | Recent context only | Keeps last K exchanges |
| **ConversationTokenBufferMemory** | Cost control | Limits by token count |
| **ConversationSummaryBufferMemory** | Long conversations | Summarizes old exchanges |

### Key Insights:

1. **LLMs are stateless** - Memory must be explicitly managed
2. **Token limits map to costs** - More tokens = higher API costs
3. **Summarization preserves context** - Keeps important info while managing size
4. **Choose memory type based on use case** - Chat vs knowledge extraction vs cost control

### Additional Memory Types (Not Covered):
- **Vector Store Memory** - Uses embeddings for semantic search
- **Entity Memory** - Tracks specific people/places/things
- **Combined Memory** - Use multiple memory types together

### Production Tip:
Store full conversations in a database (SQL/NoSQL) for auditing, then use LangChain memory for runtime context management.
